Later steps..

Determine land classification categories based on Copernicus Global Land Cover:

1. Snow and Ice
2. Ariculture
3. Urban
4. Open Forest
5. Permanent Water Bodies

Set quota per type when sampling.


In [8]:
import base64

import logging
import random
import time
import json

import ee
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm

import os
import pyogrio

import matplotlib.pyplot as plt
from umap import UMAP
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.ticker import MaxNLocator
from sklearn.manifold import TSNE


n = 2000
data_path = f"data/{n}_sampled_classified_embeddings.geojson"


# util function to save geojson while serializing embeddings as base64 strings
def save_geojson(data, out_path):
    df = pd.DataFrame(data)

    if "embedding" in df.columns:
        df["embedding"] = df["embedding"].apply(
            lambda arr: base64.b64encode(json.dumps(arr).encode("utf8")).decode("ascii")
        )

    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(lon, lat) for lon, lat in zip(df.lon, df.lat)],
        crs="EPSG:4326",
    )

    gdf.to_file(out_path, driver="GeoJSON")

    print(f"Saved {len(gdf)} rows of the GeoDataFrame to {out_path}")

## Data Generation: AlphaEarth Embeddings, Copernicus Land Classifications.

Use Google Earth Engine to sample n random locations globally.

For each sampled location:

- Get the land classificaiton from `COPERNICUS/Landcover/100m/Proba-V-C3/Global` Image collection.
- Get the satellite embedding from `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` Image collection.

If there are any issues with either of these, skip it and sample a new one.

Use the year 2019 (most recent Copernicus dataset).

The land classification dataset is at 100m resolution while the satellite embedding dataset is at 10m resolution. The geodata will be stored per point, not per region.

Store the data at each sampled point in a geo dataframe: `location (lat/lon) | classification | embedding_64d`
The 64d embeddings are serialized in a single column.

Write this to a geojson file: `data/{n}_sampled_classified_embeddings.geojson.`


In [9]:
"""
Generate sampled land points with Copernicus landcover and AlphaEarth 64-band embeddings
"""

year = 2019
embed_scale = 10


def init_ee():
    try:
        ee.Initialize(project="gsapp-map")
    except Exception:
        print("Earth Engine not initialized. Attempting authentication...")
        ee.Authenticate()
        ee.Initialize()


def sample_point(lat, lon, land_img, alphaearth, band_names):
    """(kept for single-point debugging) Sample Copernicus landcover and AlphaEarth embedding at a single point.
    Returns (classification, embedding_list) or (None, None) on failure.
    """
    pt = ee.Geometry.Point([lon, lat])

    try:
        # Sample land classification (scale 100 m)
        land_sample = (
            land_img.sampleRegions(
                collection=ee.FeatureCollection([ee.Feature(pt)]),
                scale=100,
                geometries=False,
            )
            .first()
            .getInfo()
        )
        props = land_sample.get("properties", {}) if land_sample else {}
        classification = props.get("discrete_classification")

        # Sample embedding (scale embed_scale m)
        emb_sample_fc = alphaearth.select(band_names).sampleRegions(
            collection=ee.FeatureCollection([ee.Feature(pt)]),
            scale=embed_scale,
            geometries=False,
        )
        emb_feat = emb_sample_fc.first().getInfo()
        emb_props = emb_feat.get("properties", {}) if emb_feat else {}
        embedding = [float(emb_props.get(b, float("nan"))) for b in band_names]

        # Validate embedding
        if any(np.isnan(embedding)):
            return None, None

        return classification, embedding
    except Exception as e:
        logging.debug("GEE sampling failed for point (%s,%s): %s", lat, lon, e)
        return None, None


def random_land_samples(
    n=100, year=2019, embed_scale=10, batch_size=50, max_attempts=100000
):
    """Collect up to `n` valid samples using batched GEE requests.

    - `batch_size` controls how many candidate points are queried in each GEE request.
    - Each batch adds at most `batch_size` attempts toward `max_attempts`.
    - Returned samples preserve lat/lon and include 'classification' and 'embedding'.
    """
    results = []
    attempts = 0
    batch_no = 0

    # Load image collections once
    landcol = ee.ImageCollection("COPERNICUS/Landcover/100m/Proba-V-C3/Global").filter(
        ee.Filter.calendarRange(year, year, "year")
    )
    land_img = landcol.first()

    alphaearth = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filter(ee.Filter.calendarRange(year, year, "year"))
        .mosaic()
    )

    band_names = [f"A{str(i).zfill(2)}" for i in range(64)]

    pbar = tqdm(total=n, desc="Collecting samples")

    while len(results) < n and attempts < max_attempts:
        batch_no += 1
        remaining = n - len(results)
        this_batch = min(batch_size, remaining, max_attempts - attempts)
        if this_batch <= 0:
            break

        # Generate candidate points for this batch and attach batch index
        batch_points = []
        features = []
        for i in range(this_batch):
            lat = random.uniform(-60, 80)
            lon = random.uniform(-180, 180)
            batch_points.append((lat, lon))
            feat = ee.Feature(ee.Geometry.Point([lon, lat]), {"_batch_idx": i})
            features.append(feat)

        fc = ee.FeatureCollection(features)

        # Try batched requests with a small retry/backoff on failure
        retries = 0
        max_retries = 3
        success = False
        while retries <= max_retries and not success:
            try:
                land_samples = land_img.sampleRegions(
                    collection=fc, scale=100, geometries=False
                ).getInfo()
                emb_samples = (
                    alphaearth.select(band_names)
                    .sampleRegions(collection=fc, scale=embed_scale, geometries=False)
                    .getInfo()
                )
                success = True
            except Exception as e:
                retries += 1
                wait = 1.0 * (2 ** (retries - 1))
                logging.debug("Batch %s failed (retry %s): %s", batch_no, retries, e)
                time.sleep(wait)

        attempts += this_batch

        if not success:
            print(
                f"Batch {batch_no} failed after {max_retries} retries; skipping batch."
            )
            continue

        land_feats = land_samples.get("features", []) if land_samples else []
        emb_feats = emb_samples.get("features", []) if emb_samples else []

        # Map results by _batch_idx (added as property)
        land_by_idx = {}
        for f in land_feats:
            props = f.get("properties", {})
            idx = props.get("_batch_idx")
            if idx is not None:
                land_by_idx[int(idx)] = props

        emb_by_idx = {}
        for f in emb_feats:
            props = f.get("properties", {})
            idx = props.get("_batch_idx")
            if idx is not None:
                emb_by_idx[int(idx)] = props

        # Iterate through batch in original order and collect valid samples
        collected_in_batch = 0
        for i, (lat, lon) in enumerate(batch_points):
            land_props = land_by_idx.get(i)
            emb_props = emb_by_idx.get(i)
            if not land_props or not emb_props:
                continue

            classification = land_props.get("discrete_classification")
            embedding = [emb_props.get(b) for b in band_names]
            # Ensure numeric conversion and skip NaNs
            try:
                embedding = [
                    float(v) if v is not None else float("nan") for v in embedding
                ]
            except Exception:
                continue

            if any(np.isnan(embedding)):
                continue

            results.append(
                {
                    "lat": lat,
                    "lon": lon,
                    "classification": classification,
                    "embedding": embedding,
                }
            )
            collected_in_batch += 1
            pbar.update(1)

            if len(results) >= n:
                break

        print(
            f"Batch {batch_no}: attempted {this_batch}, collected {collected_in_batch}, total {len(results)}/{n}"
        )

    pbar.close()
    return results


if os.path.exists(data_path):
    print(f"Data file {data_path} already exists. Skipping sample collection.")
else:
    init_ee()
    samples = random_land_samples(n, year, embed_scale, batch_size=100)
    if not samples:
        print("No samples collected. Exiting.")
    save_geojson(samples, data_path)

Data file data/2000_sampled_classified_embeddings.geojson already exists. Skipping sample collection.


In [10]:
# Helper function to add UN subregion info to data points


def add_subregion(gdf_points):
    # Load countries polygons
    gdf_countries = gpd.read_file(
        "data/raw/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp"
    )

    # Load UNSD M49 CSV
    m49 = pd.read_csv("data/raw/unsd.csv", sep=";")  # adjust path
    # Keep only the columns we need
    m49_subregion = m49[
        ["ISO-alpha3 Code", "Sub-region Code", "Sub-region Name"]
    ].copy()
    m49_subregion.rename(columns={"ISO-alpha3 Code": "SOV_A3"}, inplace=True)

    # Make sure CRS matches
    if gdf_points.crs != gdf_countries.crs:
        gdf_points = gdf_points.to_crs(gdf_countries.crs)

    # Spatial join points -> countries to get SOV_A3
    gdf_merged = gpd.sjoin(
        gdf_points,
        gdf_countries[["geometry", "SOV_A3"]],
        how="left",
        predicate="within",
    )

    # Merge with M49 subregion info
    gdf_merged = gdf_merged.merge(m49_subregion, on="SOV_A3", how="left")

    # Replace NaN with safe defaults
    gdf_merged["Sub-region Code"].fillna(-1, inplace=True)  # -1 for unknown
    gdf_merged["Sub-region Name"].fillna("Unknown", inplace=True)

    # Rename columns
    gdf_merged.rename(
        columns={
            "Sub-region Code": "subregion_code",
            "Sub-region Name": "subregion_name",
        },
        inplace=True,
    )

    return gdf_merged

## Data Processing: Dimension Reduction

Read the data from `data/{n}_sampled_classified_embeddings.geojson` file and load as a gdf.

Compress the 64d embedding to 2d and 3d using UMAP and t-SNE. Add new columns to the gdf.

Add subregion info if not present based on the UN's standard codes for statistical use (M49) - located in `data/raw/...`
https://unstats.un.org/unsd/methodology/m49/

Save this to `data/{n}_sampled_classified_embeddings.geojson`.


In [11]:
force_run = False


def load_gdf(path):
    gdf = gpd.read_file(path)
    print(gdf.head)
    return gdf


def extract_embeddings(gdf):
    # embeddings assumed stored as arrays in the property
    emb_list = gdf["embedding"].apply(lambda x: np.array(x, dtype=np.float32))
    emb_arr = np.vstack(emb_list.values)
    return emb_arr


def run_umap(emb_arr, n_components=2, random_state=42):
    um = UMAP(n_components=n_components, random_state=random_state)
    return um.fit_transform(emb_arr)


def run_tsne(emb_arr, n_components=2, random_state=42):
    ts = TSNE(n_components=n_components, random_state=random_state, init="random")
    return ts.fit_transform(emb_arr)


gdf = pyogrio.read_dataframe(data_path)

gdf["embedding"] = gdf["embedding"].apply(lambda v: json.loads(base64.b64decode(v)))

emb_arr = extract_embeddings(gdf)


# UMAP 1D
if force_run or "umap_1d_x" not in gdf.columns:
    print("Running UMAP 64->1 ...")
    um1 = run_umap(emb_arr, n_components=1)
    gdf["umap_1d_x"] = um1[:, 0]

# UMAP 2D
if force_run or "umap_2d_x" not in gdf.columns or "umap_2d_y" not in gdf.columns:
    print("Running UMAP 64->2 ...")
    um2 = run_umap(emb_arr, n_components=2)
    gdf["umap_2d_x"] = um2[:, 0]
    gdf["umap_2d_y"] = um2[:, 1]

# UMAP 3D
if (
    force_run
    or "umap_3d_x" not in gdf.columns
    or "umap_3d_y" not in gdf.columns
    or "umap_3d_z" not in gdf.columns
):
    print("Running UMAP 64->3 ...")
    um3 = run_umap(emb_arr, n_components=3)
    gdf["umap_3d_x"] = um3[:, 0]
    gdf["umap_3d_y"] = um3[:, 1]
    gdf["umap_3d_z"] = um3[:, 2]

# t-SNE 1D
if force_run or "tsne_1d_x" not in gdf.columns:
    print("Running t-SNE 64->1 ...")
    ts1 = run_tsne(emb_arr, n_components=1)
    gdf["tsne_1d_x"] = ts1[:, 0]

# t-SNE 2D
if force_run or "tsne_2d_x" not in gdf.columns or "tsne_2d_y" not in gdf.columns:
    print("Running t-SNE 64->2 ...")
    ts2 = run_tsne(emb_arr, n_components=2)
    gdf["tsne_2d_x"] = ts2[:, 0]
    gdf["tsne_2d_y"] = ts2[:, 1]

# t-SNE 3D
if (
    force_run
    or "tsne_3d_x" not in gdf.columns
    or "tsne_3d_y" not in gdf.columns
    or "tsne_3d_z" not in gdf.columns
):
    print("Running t-SNE 64->3 ...")
    ts3 = run_tsne(emb_arr, n_components=3)
    gdf["tsne_3d_x"] = ts3[:, 0]
    gdf["tsne_3d_y"] = ts3[:, 1]
    gdf["tsne_3d_z"] = ts3[:, 2]

# Add subregion info if not present
if (
    "SOV_A3" not in gdf.columns
    or "subregion_code" not in gdf.columns
    or "subregion_name" not in gdf.columns
):
    print("Adding subregion info ...")
    gdf = add_subregion(gdf)

save_geojson(gdf, data_path)
print("Saved output GeoJSON to", data_path)

Running UMAP 64->1 ...


/Users/emadsen/miniconda3/envs/geo/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running t-SNE 64->1 ...
Saved 2000 rows of the GeoDataFrame to data/2000_sampled_classified_embeddings.geojson
Saved output GeoJSON to data/2000_sampled_classified_embeddings.geojson


In [12]:
# Rescale coordinates to [-50, 50]
def rescale_to_range(col, target_min=-50, target_max=50):
    """Min-max normalize a column to [target_min, target_max]"""
    col_min = col.min()
    col_max = col.max()
    if col_max == col_min:
        return col.copy()
    return target_min + (col - col_min) / (col_max - col_min) * (
        target_max - target_min
    )


# Rescale UMAP 2D
gdf["umap_2d_x"] = rescale_to_range(gdf["umap_2d_x"])
gdf["umap_2d_y"] = rescale_to_range(gdf["umap_2d_y"])

# Rescale UMAP 3D
gdf["umap_3d_x"] = rescale_to_range(gdf["umap_3d_x"])
gdf["umap_3d_y"] = rescale_to_range(gdf["umap_3d_y"])
gdf["umap_3d_z"] = rescale_to_range(gdf["umap_3d_z"])

# Rescale t-SNE 2D
gdf["tsne_2d_x"] = rescale_to_range(gdf["tsne_2d_x"])
gdf["tsne_2d_y"] = rescale_to_range(gdf["tsne_2d_y"])

# Rescale t-SNE 3D
gdf["tsne_3d_x"] = rescale_to_range(gdf["tsne_3d_x"])
gdf["tsne_3d_y"] = rescale_to_range(gdf["tsne_3d_y"])
gdf["tsne_3d_z"] = rescale_to_range(gdf["tsne_3d_z"])

# Save updated GeoJSON
save_geojson(gdf, data_path)
print("Rescaled coordinates to [-100, 100] and saved to", data_path)

Saved 2000 rows of the GeoDataFrame to data/2000_sampled_classified_embeddings.geojson
Rescaled coordinates to [-100, 100] and saved to data/2000_sampled_classified_embeddings.geojson


## Plot

Create scatter plots across the following parameters:

- dimension reduction algorithm [UMAP, t-SNE]
- dimensionality [2d, 3d]
- color map [land classification, sub-region]

Save the plots in `point-clouds/`


In [13]:
LAND_CLASSIFICATION_LEGEND = {
    0: ("#282828", "Unknown / No Data"),
    20: ("#ffbb22", "Shrubs"),
    30: ("#ffff4c", "Herbaceous vegetation"),
    40: ("#f096ff", "Cultivated / Agriculture"),
    50: ("#fa0000", "Urban / Built-up"),
    60: ("#b4b4b4", "Bare / Sparse vegetation"),
    70: ("#3ed8d3", "Snow and Ice"),
    80: ("#0032c8", "Permanent water bodies"),
    90: ("#0096a0", "Herbaceous wetland"),
    100: ("#fae6a0", "Moss & Lichen"),
    111: ("#58481f", "Closed forest – evergreen needleleaf"),
    112: ("#009900", "Closed forest – evergreen broadleaf"),
    113: ("#70663e", "Closed forest – deciduous needleleaf"),
    114: ("#00cc00", "Closed forest – deciduous broadleaf"),
    115: ("#4e751f", "Closed forest – mixed"),
    116: ("#007800", "Closed forest – other"),
    121: ("#666000", "Open forest – evergreen needleleaf"),
    122: ("#8db400", "Open forest – evergreen broadleaf"),
    123: ("#8d7400", "Open forest – deciduous needleleaf"),
    124: ("#a0dc00", "Open forest – deciduous broadleaf"),
    125: ("#929900", "Open forest – mixed"),
    126: ("#648c00", "Open forest – other"),
    200: ("#000080", "Oceans / Seas"),
}

SUBREGION_LEGEND = {
    0: ("#282828", "Unknown / No Data"),
    15.0: ("#d65e27", "Northern Africa"),
    202.0: ("#e03c3c", "Sub-Saharan Africa"),
    419.0: ("#885a48", "Latin America and the Caribbean"),
    21.0: ("#D59124", "Northern America"),
    143.0: ("#6ebe61", "Central Asia"),
    30.0: ("#4d7953", "Eastern Asia"),
    35.0: ("#62a8c4", "South-eastern Asia"),
    34.0: ("#355a9e", "Southern Asia"),
    145.0: ("#1e98c0", "Western Asia"),
    151.0: ("#bcbc65", "Eastern Europe"),
    154.0: ("#a3d660", "Northern Europe"),
    39.0: ("#e9dc49", "Southern Europe"),
    155.0: ("#fff27c", "Western Europe"),
    53.0: ("#9182b6", "Australia and New Zealand"),
    54.0: ("#875ca8", "Melanesia"),
    57.0: ("#f899ea", "Micronesia"),
    61.0: ("#ff9896", "Polynesia"),
}


def plot_2d(
    gdf,
    xcol,
    ycol,
    title,
    color_map,
    class_col,
    out,
):
    os.makedirs(os.path.dirname(out), exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 6))

    classes = gdf[class_col].unique()

    for c in classes:
        sub = gdf[gdf[class_col] == c]

        # Use fallback if class not in table
        color, label = color_map.get(int(c), ("#FFFFFF", f"Class {c}"))

        ax.scatter(sub[xcol], sub[ycol], color=color, label=label, s=10)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_frame_on(False)
    ax.set_xticks([])
    ax.set_yticks([])

    ax.legend(
        title="Land Cover",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out, dpi=150)
    plt.close()


def plot_3d(
    gdf,
    xcol,
    ycol,
    zcol,
    title,
    color_map,
    class_col,
    out,
):
    os.makedirs(os.path.dirname(out), exist_ok=True)

    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection="3d")

    classes = gdf[class_col].unique()

    for c in classes:
        sub = gdf[gdf[class_col] == c]

        color, label = color_map.get(int(c), ("#FFFFFF", f"Class {c}"))  # fallback

        ax.scatter(sub[xcol], sub[ycol], sub[zcol], color=color, s=10, label=label)

    # --------- GRID LINES ---------
    ax.grid(True)
    light_gray = (0.85, 0.85, 0.85, 1)
    ax.xaxis._axinfo["grid"]["color"] = light_gray
    ax.yaxis._axinfo["grid"]["color"] = light_gray
    ax.zaxis._axinfo["grid"]["color"] = light_gray

    # --------- CONSISTENT NUMBER OF GRID LINES ---------
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_major_locator(MaxNLocator(nbins=5))  # 5 lines per axis

    # --------- REMOVE TICK LABELS ---------
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    ax.tick_params(axis="both", which="both", length=0)  # remove tick lines

    ax.xaxis.pane.set_edgecolor("none")
    ax.yaxis.pane.set_edgecolor("none")
    ax.zaxis.pane.set_edgecolor("none")
    ax.xaxis.pane.set_facecolor((1, 1, 1, 0))  # optional: make panes transparent
    ax.yaxis.pane.set_facecolor((1, 1, 1, 0))
    ax.zaxis.pane.set_facecolor((1, 1, 1, 0))

    # --------- Titles & Labels ---------
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.set_title(title)

    # --------- LEGEND ---------
    ax.legend(
        title="Land Cover",
        bbox_to_anchor=(1, 1),
        fontsize=8,
        title_fontsize=10,
    )

    fig.subplots_adjust(left=0.05, right=0.7, top=0.95, bottom=0.05)

    plt.savefig(out, dpi=150)
    plt.close()


# UMAP 2d by land classification
if not os.path.exists(f"output/{n}_umap_2d_landclassification.png"):
    plot_2d(
        gdf,
        "umap_2d_x",
        "umap_2d_y",
        title="UMAP 2d: AlphaEarth Embeddings by Land Classification",
        color_map=LAND_CLASSIFICATION_LEGEND,
        class_col="classification",
        out=f"output/{n}_umap_2d_landclassification.png",
    )

# UMAP 3d by land classification
if not os.path.exists(f"output/{n}_umap_3d_landclassification.png"):
    plot_3d(
        gdf,
        "umap_3d_x",
        "umap_3d_y",
        "umap_3d_z",
        title="UMAP 3d: AlphaEarth Embeddings by Land Classification",
        color_map=LAND_CLASSIFICATION_LEGEND,
        class_col="classification",
        out=f"output/{n}_umap_3d_landclassification.png",
    )

# t-SNE 2d by land classification
if not os.path.exists(f"output/{n}_tsne_2d_landclassification.png"):
    plot_2d(
        gdf,
        "tsne_2d_x",
        "tsne_2d_y",
        title="t-SNE 2d: AlphaEarth Embeddings by Land Classification",
        color_map=LAND_CLASSIFICATION_LEGEND,
        class_col="classification",
        out=f"output/{n}_tsne_2d_landclassification.png",
    )

# t-SNE 3d by land classification
if not os.path.exists(f"output/{n}_tsne_3d_landclassification.png"):
    plot_3d(
        gdf,
        "tsne_3d_x",
        "tsne_3d_y",
        "tsne_3d_z",
        title="t-SNE 3d: AlphaEarth Embeddings by Land Classification",
        color_map=LAND_CLASSIFICATION_LEGEND,
        class_col="classification",
        out=f"output/{n}_tsne_3d_landclassification.png",
    )

# UMAP 2d by subregion
if not os.path.exists(f"output/{n}_umap_2d_subregion.png"):
    plot_2d(
        gdf,
        "umap_2d_x",
        "umap_2d_y",
        title="UMAP 2d: AlphaEarth Embeddings by Subregion",
        color_map=SUBREGION_LEGEND,
        class_col="subregion_code",
        out=f"output/{n}_umap_2d_subregion.png",
    )

# UMAP 3d by subregion
if not os.path.exists(f"output/{n}_umap_3d_subregion.png"):
    plot_3d(
        gdf,
        "umap_3d_x",
        "umap_3d_y",
        "umap_3d_z",
        title="UMAP 3d: AlphaEarth Embeddings by Subregion",
        color_map=SUBREGION_LEGEND,
        class_col="subregion_code",
        out=f"output/{n}_umap_3d_subregion.png",
    )

# t-SNE 2d by subregion
if not os.path.exists(f"output/{n}_tsne_2d_subregion.png"):
    plot_2d(
        gdf,
        "tsne_2d_x",
        "tsne_2d_y",
        title="t-SNE 2d: AlphaEarth Embeddings by Subregion",
        color_map=SUBREGION_LEGEND,
        class_col="subregion_code",
        out=f"output/{n}_tsne_2d_subregion.png",
    )

# t-SNE 3d by subregion
if not os.path.exists(f"output/{n}_tsne_3d_subregion.png"):
    plot_3d(
        gdf,
        "tsne_3d_x",
        "tsne_3d_y",
        "tsne_3d_z",
        title="t-SNE 3d: AlphaEarth Embeddings by Subregion",
        color_map=SUBREGION_LEGEND,
        class_col="subregion_code",
        out=f"output/{n}_tsne_3d_subregion.png",
    )


print("Saved plots to output/...")

Saved plots to output/...
